# 무작위로 지우기: Dropout, DropEdge, DropNode

깊은 GNN 은 파라미터가 많은데 레이블은 적어서 과적합하고, 이웃을 너무 많이 섞어서 표현이 비슷해져요. 엣지를 에폭마다 조금씩 지우면 두 문제를 한 번에 덜 수 있어요. 앞부분은 지우기와 차수 다시 세기를 numpy 없이 순수 파이썬으로 직접 짜 보고, 뒷부분은 슬라이드 p.46, p.48, p.49 에 실제로 찍혀 있는 코드를 그대로 옮겨 Colab 에서 돌려요. 5주차는 실습 노트북이 없어서 슬라이드 코드만 썼어요.

**하는 법**
1. 위 메뉴 **런타임 > 런타임 유형 변경** 은 CPU 그대로 두어도 돼요.
2. 문제마다 **내 코드 칸**을 채우고 실행(Shift+Enter)한 뒤, 바로 아래 **채점 칸**을 실행해요.
3. `통과!` 가 나오면 성공, 빨간 `AssertionError` 가 나오면 마지막 줄의 한국어 안내를 읽고 고쳐요.
4. 막히면 **힌트**를 한 단계씩 펼쳐 보고, 그래도 안 되면 **정답 보기**를 펼쳐요.

## 준비: 필요한 도구 설치 (처음 한 번만 실행)

In [ ]:
!pip install -q torch_geometric

## 1. PyG 의 dropout_edge 써 보기  (따라 치기)

슬라이드 p.48 에 나오는 `dropout_edge` 를 직접 불러 봐요. 경계값으로 동작을 확인해요.

- `from torch_geometric.utils import dropout_edge` 예요. 슬라이드 제목에는 `torch.nn.Dropout` 이라고 적혀 있지만, 실제로 쓰는 함수는 `torch_geometric.utils` 의 것이에요. 슬라이드 제목이 앞 장에서 그대로 복사된 것으로 보여요.
- `dropout_edge` 는 값을 **두 개** 돌려줘요. 지운 뒤의 `edge_index` 와 어떤 엣지가 남았는지 적힌 마스크예요.
- `p=0.0` 이면 하나도 안 지우고 `p=1.0` 이면 전부 지워요.
- `training=False` 를 주면 확률과 상관없이 아무것도 안 지워요. 추론할 때는 원래 그래프를 써야 하기 때문이에요.
- `.size(1)` 은 두 번째 축의 길이, 곧 엣지 항목 수예요.
- 같은 이름의 `dropout_node` 도 있어요. 그쪽은 노드를 지우고 값을 세 개 돌려줘요.

아래 코드를 **보면서 직접 쳐 보세요** (복사하지 말고요):

```python
import torch
from torch_geometric.utils import dropout_edge

edge_index = torch.tensor([[0, 1, 1, 2, 2, 3],
                           [1, 0, 2, 1, 3, 2]])
kept_all, mask_all = dropout_edge(edge_index, p=0.0)
kept_none, mask_none = dropout_edge(edge_index, p=1.0)
eval_ei, _ = dropout_edge(edge_index, p=0.5, training=False)
print(kept_all.size(1), kept_none.size(1), eval_ei.size(1))
```

<details><summary>생각 순서 보기 (힌트보다 먼저)</summary>

1. 입력은 2 x E 짜리 정수 텐서예요.
2. 확률이 0 과 1 인 경계부터 확인하면 함수가 어느 쪽 확률인지 헷갈리지 않아요.
3. 6, 0, 6 이 차례로 나와야 해요.
4. 학습이 아닐 때 그래프가 온전히 남는다는 점을 눈으로 확인하세요.

슈도코드

```text
edge_index 텐서를 만든다
p = 0 으로 한 번 지워 본다
p = 1 로 한 번 지워 본다
training = False 로 한 번 더 해 본다
남은 엣지 수 셋을 출력한다
```

</details>

<details><summary>힌트 1</summary>

import 줄부터 그대로 쳐 보세요

</details>

<details><summary>힌트 2</summary>

돌려주는 값이 두 개예요

</details>

<details><summary>힌트 3</summary>

size(1) 로 엣지 수를 세요

</details>

원본: L5 슬라이드 p.48 (dropout_edge 예시 코드)


In [ ]:
# 여기에 코드를 쳐 보세요


In [ ]:
# 채점 칸: 위 칸을 실행한 다음 이 칸을 실행하세요
assert "kept_all" in globals() and "kept_none" in globals(), "kept_all 과 kept_none 을 만들어야 해요"
assert kept_all.size(1) == 6, f"p = 0 이면 엣지 6 개가 다 남아야 해요. 지금은 {kept_all.size(1)} 개예요"
assert kept_none.size(1) == 0, f"p = 1 이면 엣지가 하나도 안 남아야 해요. 지금은 {kept_none.size(1)} 개예요"
assert eval_ei.size(1) == 6, "training=False 면 확률과 상관없이 전부 남아야 해요"
assert bool(mask_all.all()), "p = 0 일 때 마스크는 전부 True 여야 해요"
assert mask_all.numel() == 6, "마스크 길이는 엣지 수와 같아야 해요"
print("통과! 슬라이드 p.49 의 학습 루프가 이 함수를 에폭마다 불러요. 지금 그 한 줄을 이해한 거예요.")

<details><summary>정답 보기</summary>

```python
import torch
from torch_geometric.utils import dropout_edge

edge_index = torch.tensor([[0, 1, 1, 2, 2, 3],
                           [1, 0, 2, 1, 3, 2]])
kept_all, mask_all = dropout_edge(edge_index, p=0.0)
kept_none, mask_none = dropout_edge(edge_index, p=1.0)
eval_ei, _ = dropout_edge(edge_index, p=0.5, training=False)
print(kept_all.size(1), kept_none.size(1), eval_ei.size(1))
```

</details>

## 2. Dropout 을 넣은 GCN 클래스  (직접 짜기)

슬라이드 p.46 의 GCN 클래스를 직접 짜요. 두 합성곱 층 **사이** 에 Dropout 이 들어가요.

- `class GCN(torch.nn.Module):` 는 PyTorch 가 준 기본 틀을 물려받는다는 뜻이에요.
- `super(GCN, self).__init__()` 를 빠뜨리면 층을 속성으로 넣어도 PyTorch 가 파라미터를 못 찾아요.
- `torch.nn.Dropout(p=drop)` 은 학습 중에 입력 텐서의 원소를 확률 p 로 0 으로 만들어요. 그래프가 아니라 **특징** 을 흔드는 것이에요.
- `F.log_softmax(x, dim=1)` 의 `dim=1` 은 '각 노드의 클래스 방향으로' 라는 뜻이에요. 0 으로 두면 노드 사이에서 확률을 나눠 완전히 틀린 계산이 돼요.
- 슬라이드의 `forward` 는 쓰지 않는 인자 `g` 와 `Kindices` 를 받아요. 원본 그대로 옮겨 두었어요.

<details><summary>생각 순서 보기 (힌트보다 먼저)</summary>

1. 입력은 노드 특징 x 와 edge_index 이고 출력은 노드마다 클래스 점수예요.
2. 층 두 개와 Dropout 하나, 그리고 중간 결과를 담을 h 가 필요해요.
3. Dropout 자리가 중요해요. ReLU 를 지난 뒤, 두 번째 합성곱 앞이에요.
4. `self.h` 는 log_softmax 를 씌우기 **전** 값을 담아요. 시각화에 쓰려고 남겨 둔 거예요.
5. 노드 4 개, 특징 3 개, 클래스 2 개로 돌려서 출력이 4 x 2 인지 확인하세요.

슈도코드

```text
nn.Module 을 물려받는 클래스를 연다
__init__ 에서 부모를 먼저 부른다
합성곱 두 개와 Dropout 하나를 속성으로 만든다
h 는 None 으로 둔다
forward 에서 conv1 과 ReLU 를 지난다
Dropout 을 지난다
conv2 를 지나 self.h 에 담는다
log_softmax 를 씌워 돌려준다
```

</details>

<details><summary>힌트 1</summary>

super().__init__() 를 먼저 부르세요

</details>

<details><summary>힌트 2</summary>

Dropout 은 ReLU 다음, conv2 앞이에요

</details>

<details><summary>힌트 3</summary>

마지막 줄은 return F.log_softmax(x, dim=1) 이에요

</details>

원본: L5 슬라이드 p.46 (Dropout 예시 코드)


In [ ]:
import torch
import torch.nn.functional as F
from torch_geometric.nn import GCNConv

# 슬라이드 p.46 의 GCN 클래스를 그대로 만들어요
# 1. __init__ 에서 conv1, conv2, drop, h 네 개를 만들어요
#    conv1 = GCNConv(num_features, dim), conv2 = GCNConv(dim, num_classes)
#    drop = torch.nn.Dropout(p=drop), h = None
# 2. forward(self, x, edge_index, g, Kindices) 에서
#    conv1 에 ReLU, 그 다음 drop, 그 다음 conv2 순서로 통과시켜요
# 3. self.h 에 conv2 결과를 담고 F.log_softmax(x, dim=1) 을 돌려줘요
class GCN(torch.nn.Module):
    ...


In [ ]:
# 채점 칸: 위 칸을 실행한 다음 이 칸을 실행하세요
assert "GCN" in globals(), "GCN 클래스를 만들어야 해요"
m = GCN(3, 2)
assert isinstance(m, torch.nn.Module), "torch.nn.Module 을 물려받아야 해요"
assert len(list(m.parameters())) > 0, "파라미터가 하나도 없어요. super().__init__() 를 불렀는지 확인하세요"
assert isinstance(m.drop, torch.nn.Dropout), "self.drop 은 torch.nn.Dropout 이어야 해요"
assert abs(m.drop.p - 0.5) < 1e-9, f"기본 drop 은 0.5 예요. 지금은 {m.drop.p} 이에요"
assert m.conv1.in_channels == 3 and m.conv1.out_channels == 16, "conv1 은 GCNConv(num_features, dim) 이어야 해요"
assert m.conv2.in_channels == 16 and m.conv2.out_channels == 2, "conv2 는 GCNConv(dim, num_classes) 여야 해요"
torch.manual_seed(0)
x = torch.randn(4, 3)
ei = torch.tensor([[0, 1, 1, 2, 2, 3], [1, 0, 2, 1, 3, 2]])
m.eval()
out = m(x, ei, None, None)
assert tuple(out.shape) == (4, 2), f"출력은 노드 4 개 x 클래스 2 개여야 해요. 지금은 {tuple(out.shape)} 이에요"
assert m.h is not None and tuple(m.h.shape) == (4, 2), "self.h 에 conv2 의 결과를 담아야 해요"
assert torch.allclose(out.exp().sum(dim=1), torch.ones(4), atol=1e-5), "log_softmax 의 dim 은 1 이어야 해요 (노드마다 클래스 확률의 합이 1)"
m2 = GCN(5, 3, dim=8, drop=0.2)
assert m2.conv1.out_channels == 8 and abs(m2.drop.p - 0.2) < 1e-9, "dim 과 drop 을 바꿔도 반영되어야 해요"
print("통과! 이 클래스에 슬라이드 p.49 의 `train_dropEdges` 를 붙이면 Dropout 과 DropEdge 를 같이 쓰는 모델이 돼요.")

<details><summary>정답 보기</summary>

```python
import torch
import torch.nn.functional as F
from torch_geometric.nn import GCNConv


class GCN(torch.nn.Module):
    def __init__(self, num_features, num_classes, dim=16, drop=0.5):
        super(GCN, self).__init__()
        self.conv1 = GCNConv(num_features, dim)
        self.conv2 = GCNConv(dim, num_classes)
        self.drop = torch.nn.Dropout(p=drop)
        self.h = None

    def forward(self, x, edge_index, g, Kindices):
        x = F.relu(self.conv1(x, edge_index))
        x = self.drop(x)
        x = self.conv2(x, edge_index)
        self.h = x
        return F.log_softmax(x, dim=1)

```

</details>

## 3. DropEdge 학습 한 걸음 직접 짜기  (직접 짜기)

슬라이드 p.49 의 `train_dropEdges` 안쪽 한 에폭을 함수 하나로 만들어요.

- `optimizer.zero_grad()` 는 지난 기울기를 지워요. 빠뜨리면 기울기가 계속 더해져 학습이 망가져요.
- `edge_index1, _ = dropout_edge(...)` 에서 `_` 는 '쓰지 않을 값' 이라는 뜻이에요. 마스크는 필요 없어요.
- 모델에는 원래 `data.edge_index` 가 아니라 **지운** `edge_index1` 을 넣어야 해요. 이 한 글자가 DropEdge 의 전부예요.
- `loss.backward()` 로 기울기를 구하고 `optimizer.step()` 으로 가중치를 고쳐요. 순서를 바꾸면 안 돼요.
- `loss.item()` 은 텐서에서 파이썬 숫자 하나를 꺼내요. 기록용으로 쓸 때는 꼭 이렇게 꺼내야 메모리가 안 쌓여요.

<details><summary>생각 순서 보기 (힌트보다 먼저)</summary>

1. 입력은 모델, 옵티마이저, 손실 함수, 특징, 엣지, 정답, 확률이에요. 출력은 손실 숫자 하나예요.
2. 순서가 전부인 문제예요. 지우기는 모델 호출 앞, zero_grad 는 맨 앞이에요.
3. `p = 0` 으로 주면 원래 그래프로 학습하는 것과 같아야 해요.
4. 같은 함수를 두 번 부르면 손실이 줄어드는지 확인해 보면 학습이 실제로 되는지 알 수 있어요.
5. `return` 이 `optimizer.step()` 뒤에 있어야 해요.

슈도코드

```text
지난 기울기를 지운다
엣지를 확률 p 로 무작위로 지운다
지운 엣지로 모델을 돌려 예측을 얻는다
예측과 정답으로 손실을 구한다
기울기를 구한다
가중치를 고친다
손실 숫자를 돌려준다
```

</details>

<details><summary>힌트 1</summary>

zero_grad 를 맨 앞에 두세요

</details>

<details><summary>힌트 2</summary>

model 에 넣는 것은 edge_index1 이에요

</details>

<details><summary>힌트 3</summary>

backward 다음에 step, 그 다음에 return loss.item() 이에요

</details>

원본: L5 슬라이드 p.49 (train_dropEdges 예시 코드)


In [ ]:
import torch
from torch_geometric.utils import dropout_edge

# 1. optimizer.zero_grad() 로 지난 기울기를 지워요
# 2. dropout_edge(data_edge_index, p=p) 로 지운 edge_index1 을 얻어요 (두 번째 값은 _ 로 버려요)
# 3. 지운 edge_index1 로 model(x, edge_index1) 을 불러 out 을 얻어요
# 4. loss = criterion(out, y) 를 구하고 loss.backward(), optimizer.step() 을 해요
# 5. loss 의 숫자 값(loss.item())을 돌려주는 함수 train_step(...) 을 완성해요
def train_step(model, optimizer, criterion, x, data_edge_index, y, p):
    ...


In [ ]:
# 채점 칸: 위 칸을 실행한 다음 이 칸을 실행하세요
assert "train_step" in globals(), "train_step 함수를 만들어야 해요"
from torch_geometric.nn import GCNConv


class Tiny(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = GCNConv(3, 2)
        self.calls = []

    def forward(self, x, edge_index):
        self.calls.append(edge_index.size(1))
        return self.conv(x, edge_index)


torch.manual_seed(0)
x = torch.randn(4, 3)
ei = torch.tensor([[0, 1, 1, 2, 2, 3], [1, 0, 2, 1, 3, 2]])
y = torch.tensor([0, 1, 0, 1])
model = Tiny()
opt = torch.optim.Adam(model.parameters(), lr=0.05)
crit = torch.nn.CrossEntropyLoss()
before = model.conv.lin.weight.detach().clone()
loss0 = train_step(model, opt, crit, x, ei, y, 0.0)
assert isinstance(loss0, float), "loss.item() 으로 숫자 하나를 돌려줘야 해요"
assert model.calls and model.calls[-1] == 6, "p = 0 이면 엣지 6 개가 그대로 모델에 들어가야 해요"
assert not torch.allclose(before, model.conv.lin.weight), "가중치가 안 바뀌었어요. optimizer.step() 을 불렀는지 확인하세요"
losses = [train_step(model, opt, crit, x, ei, y, 0.0) for _ in range(40)]
assert losses[-1] < loss0, f"학습이 되면 손실이 줄어야 해요. 처음 {loss0:.4f}, 마지막 {losses[-1]:.4f}. zero_grad 와 backward 순서를 확인하세요"
model2 = Tiny()
opt2 = torch.optim.Adam(model2.parameters(), lr=0.05)
torch.manual_seed(3)
_ = [train_step(model2, opt2, crit, x, ei, y, 0.5) for _ in range(20)]
assert min(model2.calls) < 6, "p = 0.5 면 어떤 에폭에서는 엣지가 줄어야 해요. 지운 edge_index1 을 모델에 넣었는지 확인하세요"
print("통과! 슬라이드 p.49 는 이 함수를 `for epoch in range(epochs+1):` 안에서 16 번 불러요. 우리가 짠 것이 그 안쪽 한 바퀴예요.")

<details><summary>정답 보기</summary>

```python
import torch
from torch_geometric.utils import dropout_edge


def train_step(model, optimizer, criterion, x, data_edge_index, y, p):
    optimizer.zero_grad()
    edge_index1, _ = dropout_edge(data_edge_index, p=p)
    out = model(x, edge_index1)
    loss = criterion(out, y)
    loss.backward()
    optimizer.step()
    return loss.item()

```

</details>